# Eksplorasi Permintaan Mingguan dan Rancangan Fitur Model

Capstone Asah 2026, tim A26-CS010, jalur AI Engineer.

**Pertanyaan yang dijawab notebook ini:**
1. Seperti apa pola pemesanan mingguan pada data asli UMKM?
2. Fitur apa yang layak dipakai model prediksi permintaan?
3. Apakah data contoh buatan tim membantu atau justru merusak prediksi pada data asli?
4. Apakah model mengalahkan pembanding sederhana?

**Catatan privasi.** Dataset tidak disimpan di repositori. Taruh berkasnya di `data/raw/`
(folder ini diabaikan git), lalu sesuaikan `DATA_PATH` di bawah. Repositori ini publik, jadi
**hapus output sel sebelum commit** (menu *Kernel → Restart & Clear Output*), karena output
berisi angka pemesanan per minggu milik UMKM.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

# Notebook ini memakai fungsi preprocessing yang sama dengan layanan model.
ML_SERVICE = Path.cwd().parent / "ml-service"
sys.path.insert(0, str(ML_SERVICE))
from src.preprocessing import (  # noqa: E402
    add_features,
    features_for_weeks,
    load_bookings,
    load_event_weeks,
    mondays_in_month,
    to_weekly,
)

DATA_PATH = Path.cwd().parent / "data" / "raw" / "dataset_booking_kebaya_1640_lengkap.csv"

## 1. Pola pemesanan mingguan (data asli)

Baris pemesanan diringkas menjadi jumlah pemesanan per minggu (minggu mulai Senin, sama dengan
kontrak API). Minggu tanpa pemesanan diisi 0 karena memang nol, bukan data hilang.

In [ ]:
bookings = load_bookings(DATA_PATH)
events = load_event_weeks()

weekly_asli = add_features(to_weekly(bookings, only_origin="asli"), events)
weekly_semua = add_features(to_weekly(bookings), events)

# Nomor minggu dalam setahun, hanya untuk diuji di notebook ini (tidak dipakai layanan).
for frame in (weekly_asli, weekly_semua):
    frame["minggu_ke"] = frame["week_start"].dt.isocalendar().week.astype(int)

print(f"Data asli: {len(weekly_asli)} minggu, "
      f"{weekly_asli.week_start.min():%d %b %Y} sampai {weekly_asli.week_start.max():%d %b %Y}")
print(f"Kalender minggu acara terisi: {len(events)} minggu")
weekly_asli["n_bookings"].describe().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(weekly_asli["week_start"], weekly_asli["n_bookings"], width=5, color="#4C6EF5")
ax.set_title("Jumlah pemesanan per minggu, data asli")
ax.set_ylabel("Pemesanan")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

per_bulan = weekly_asli.groupby(weekly_asli["week_start"].dt.month.rename("bulan"))["n_bookings"].sum()
# Catatan: bulan terakhir bisa berisi minggu yang belum penuh, tergantung tanggal akhir data.
per_bulan.rename("pemesanan per bulan").to_frame().T

Pola yang terlihat: sebagian besar minggu sepi, tetapi ada beberapa minggu dengan lonjakan
besar. Lonjakan ini kemungkinan besar minggu wisuda atau acara, sehingga yang menentukan
permintaan adalah **jadwal acara**, bukan sekadar bulan.

## 2. Membandingkan rancangan fitur dan sumber data latih

**Metode evaluasi bergulir.** Untuk setiap minggu uji, model dilatih memakai semua minggu
sebelumnya, lalu memprediksi minggu itu. Data asli hanya sekitar 23 minggu, jadi cara ini
memberi lebih banyak titik uji daripada satu kali pemisahan, tanpa pernah memakai data masa
depan. Penilaian **selalu** pada data asli.

**Pembanding:** rata-rata minggu sebelumnya, median, dan "sama dengan minggu lalu".

In [ ]:
def preprocessor(cols):
    transformers = []
    if "bulan" in cols:
        transformers.append(("bulan", OneHotEncoder(handle_unknown="ignore"), ["bulan"]))
    numerik = [c for c in cols if c != "bulan"]
    if numerik:
        transformers.append(("numerik", StandardScaler(), numerik))
    return ColumnTransformer(transformers)


def pipeline(model, cols):
    return Pipeline([("pra", preprocessor(cols)), ("model", model)])


KONFIGURASI = {
    "Ridge, bulan + minggu_ke": (Ridge(alpha=1.0), ["bulan", "minggu_ke"]),
    "Ridge, bulan": (Ridge(alpha=1.0), ["bulan"]),
    "Random Forest, bulan + minggu_ke": (RandomForestRegressor(300, random_state=42), ["bulan", "minggu_ke"]),
    "Random Forest, bulan": (RandomForestRegressor(300, random_state=42), ["bulan"]),
}
MULAI_UJI = 10  # minggu uji pertama; sisanya menjadi titik uji


def evaluasi_bergulir(pool_latih):
    galat = {"Pembanding: rata-rata": [], "Pembanding: median": [], "Pembanding: minggu lalu": []}
    galat.update({k: [] for k in KONFIGURASI})
    for i in range(MULAI_UJI, len(weekly_asli)):
        minggu = weekly_asli["week_start"].iloc[i]
        aktual = weekly_asli["n_bookings"].iloc[i]
        riwayat = weekly_asli.iloc[:i]["n_bookings"]
        galat["Pembanding: rata-rata"].append(abs(aktual - riwayat.mean()))
        galat["Pembanding: median"].append(abs(aktual - riwayat.median()))
        galat["Pembanding: minggu lalu"].append(abs(aktual - riwayat.iloc[-1]))
        latih = pool_latih[pool_latih["week_start"] < minggu]
        uji = weekly_asli.iloc[[i]]
        for nama, (model, cols) in KONFIGURASI.items():
            pred = pipeline(clone(model), cols).fit(latih[cols], latih["n_bookings"]).predict(uji[cols])[0]
            galat[nama].append(abs(aktual - max(0.0, pred)))
    return {k: np.mean(v) for k, v in galat.items()}


hasil = pd.DataFrame({
    "MAE, latih data asli": evaluasi_bergulir(weekly_asli),
    "MAE, latih asli + contoh": evaluasi_bergulir(weekly_semua),
}).round(2)
print(f"{len(weekly_asli) - MULAI_UJI} minggu uji, prediksi 1 minggu ke depan")
hasil

Cara membaca tabel: MAE adalah rata-rata selisih prediksi terhadap jumlah pemesanan sebenarnya
per minggu. Makin kecil makin baik. Pembanding tidak dilatih, jadi angkanya sama di kedua kolom.

## 3. Uji ekstrapolasi: prediksi Desember 2026

Desember tidak ada di data asli. Uji ini memperlihatkan apa yang terjadi saat model diminta
memprediksi bulan yang belum pernah ia lihat.

In [ ]:
desember = features_for_weeks(mondays_in_month("2026-12"), events)
desember["minggu_ke"] = pd.to_datetime(mondays_in_month("2026-12")).isocalendar().week.astype(int).to_numpy()

baris = []
for sumber, pool in [("asli", weekly_asli), ("asli + contoh", weekly_semua)]:
    for nama, (model, cols) in KONFIGURASI.items():
        pred = pipeline(clone(model), cols).fit(pool[cols], pool["n_bookings"]).predict(desember[cols])
        baris.append({"sumber latih": sumber, "konfigurasi": nama,
                      "total Desember": round(float(np.clip(pred, 0, None).sum()), 1)})

print("Rujukan, total bulanan tertinggi di data asli:", int(per_bulan.max()))
pd.DataFrame(baris)

## 4. Kesimpulan dan keputusan

Kesimpulan di bawah diambil dari dataset draf (25 November 2026). **Jalankan ulang notebook ini
setelah dataset final dari tim Data Science**, lalu periksa apakah kesimpulannya masih sama.

1. **`minggu_ke` dibuang dari fitur.** Pada uji ekstrapolasi, konfigurasi dengan `minggu_ke`
   memprediksi Desember sekitar 200 pemesanan, pada Ridge maupun Random Forest, kira-kira dua kali
   bulan tersibuk di data asli. Sebabnya, lonjakan terbesar ada di ujung data, sehingga minggu-minggu
   sesudahnya dibaca sebagai kelanjutan lonjakan.
2. **Model dilatih dari data asli saja.** Menambahkan data contoh membuat MAE pada data asli
   hampir dua kali lipat, karena pola data contoh (merata sepanjang minggu) berbeda dari pola asli
   (bergerombol pada minggu acara).
3. **Model belum mengalahkan pembanding "sama dengan minggu lalu".** Tanpa informasi jadwal acara,
   fitur bulan saja tidak bisa menebak minggu mana yang akan melonjak. Ini dilaporkan apa adanya di
   README, sesuai aturan Asah.
4. **Langkah paling berdampak berikutnya adalah kalender minggu acara** (misalnya jadwal wisuda
   kampus di sekitar UMKM), diisi di `ml-service/config/event_weeks.json`. Jadwalnya wajib dari
   sumber nyata dan dicantumkan sumbernya; jadwal karangan tim tidak boleh dipakai.
5. Untuk bulan yang tidak ada di data asli (misalnya Desember), layanan tetap memberi angka tetapi
   menyertakan `note` peringatan bahwa angkanya hanya gambaran kasar.